# Parsers against each other and against the judge

Runs the current Python parser, the Scala `PeriodParser` (through `PeriodParserCli` and sbt) and
`period_new` over every string in `data/periods.jsonl`, shows where they disagree, and scores each
against the judge's answers in `data/judge-output`. Ranges are compared as inclusive day bounds
with open sides as `None`; identifiers are not compared.

In [1]:
import json
import subprocess
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path

from adapters.transformers.ebsco.production import _parse_period_or_bare_label
from adapters.transformers.ebsco.label_subdivisions import build_concept
from adapters.transformers.marc.parsers.period_new import parse as parse_new
from adapters.transformers.utils.text_utils import normalise_label

rows = [json.loads(line) for line in Path("data/periods.jsonl").open(encoding="utf-8")]
occurrences = Counter()
for row in rows:
    occurrences[(row["source"], row["path"], row["text"])] += row["n"]
print(f"{sum(occurrences.values())} strings, {len(occurrences)} distinct (source, path, text) triples")

1547871 strings, 76343 distinct (source, path, text) triples


In [2]:
OPEN = {"0001-01-01", "9999-12-31", "-9999-01-01"}


def bounds(start, end):
    return tuple(None if not d or d.split("T")[0] in OPEN else d.split("T")[0] for d in (start, end))


def python_bounds(source, path, text):
    try:
        concept = _parse_period_or_bare_label(normalise_label(text, "Period")) if path == "production" else build_concept(text, "Period")
    except Exception:
        return (None, None)
    rng = getattr(concept, "range", None)
    return bounds(rng.from_time, rng.to_time) if rng else (None, None)


def new_bounds(source, path, text):
    span = parse_new(text, source)
    return bounds(span[0].isoformat(), span[1].isoformat()) if span else (None, None)

In [100]:
scala_in = Path("data/scala-in.txt").resolve()
scala_out = Path("data/scala-out.jsonl").resolve()
scala_in.write_text("\n".join(sorted({text.replace("\n", " ") for _, _, text in occurrences})), encoding="utf-8")
sbt = subprocess.run(
    ["sbt", "-batch", f"transformer_common/Test/runMain weco.pipeline.transformer.parse.PeriodParserCli {scala_in} {scala_out}"],
    cwd="../..", capture_output=True, text=True,
)
assert sbt.returncode == 0, sbt.stdout[-2000:] + sbt.stderr[-2000:]

scala = {}
for line in scala_out.read_text(encoding="utf-8").splitlines():
    row = json.loads(line)
    rng = row["range"] or {}
    scala[row["input"]] = bounds(rng.get("from"), rng.get("to"))

PARSERS = {"python": python_bounds, "scala": lambda source, path, text: scala[text], "new": new_bounds}
#PARSERS = {"scala": lambda source, path, text: scala[text], "new": new_bounds}
results = {name: {key: parser(*key) for key in occurrences} for name, parser in PARSERS.items()}

In [101]:
def verdict(key):
    got = {name: results[name][key] for name in PARSERS}
    if len(set(got.values())) == 1:
        return "all agree"
    return "differ: " + ", ".join(sorted(name for name in PARSERS if list(got.values()).count(got[name]) == 1))


verdicts = {key: verdict(key) for key in occurrences}
summary = Counter()
for key, v in verdicts.items():
    summary[(key[1], v)] += occurrences[key]
for (path, v), n in sorted(summary.items()):
    print(f"{path:11} {v:24} {n:>9}")

genre       all agree                     7407
genre       differ: new                      2
genre       differ: new, python, scala         4
genre       differ: python                 954
production  all agree                  1258724
production  differ: new                   2825
production  differ: new, python, scala      7638
production  differ: python               86914
production  differ: scala                15006
subject     all agree                   132975
subject     differ: new                  35182
subject     differ: new, python, scala        21
subject     differ: python                 209
subject     differ: scala                   10


In [102]:
LIMIT = 100


def show(b):
    return f"{b[0] or 'open'} .. {b[1] or 'open'}" if b != (None, None) else "no range"


for key in sorted((k for k in occurrences if verdicts[k] != "all agree"), key=lambda k: -occurrences[k])[:LIMIT]:
    source, path, text = key
    print(
        f"{occurrences[key]:<7} {text!r:<50} {show(results['scala'][key]):<25} {show(results['new'][key]):<25}"
    )
    # for name in PARSERS:
    #     print(f"    {name:7} {show(results[name][key])}")

6934    '19th-20th centuries.'                             no range                  1800-01-01 .. 1999-12-31 
2577    '18th-19th centuries.'                             no range                  1700-01-01 .. 1899-12-31 
1784    'To 1500.'                                         no range                  open .. 1500-12-31       
1510    'Revolution, 1775-1783'                            no range                  1775-01-01 .. 1783-12-31 
1196    '16th-17th centuries.'                             no range                  1500-01-01 .. 1699-12-31 
1117    '17th-18th centuries.'                             no range                  1600-01-01 .. 1799-12-31 
891     '18th-20th centuries.'                             no range                  1700-01-01 .. 1999-12-31 
821     'Revolution, 1775-1783.'                           no range                  1775-01-01 .. 1783-12-31 
817     'Revolution, 1789-1799.'                           no range                  1789-01-01 .. 1799-12-31 
7

## Against the judge

`unparseable` and `ambiguous` both mean no range is right. Judge answers that are malformed or
run backwards are dropped. Precision is correct ranges over ranges produced; recall is correct
ranges over ranges the judge found.

In [104]:
index = json.load(Path("data/judge-input/index.json").open(encoding="utf-8"))
judged, malformed = {}, 0
for f in sorted(Path("data/judge-output-100-training").glob("batch-*.jsonl")):
    for line in f.read_text(encoding="utf-8").splitlines():
        if not line.strip() or line.startswith("```"):
            continue
        try:
            id_, *rest = json.loads(line)
            assert len(rest) == 5 and id_ in index
            judged[id_] = tuple(rest)
        except (ValueError, AssertionError):
            malformed += 1


def valid(outcome, start, end):
    if outcome != "range":
        return outcome in ("unparseable", "ambiguous")
    try:
        return bool(start or end) and all(date.fromisoformat(d.lstrip("-")) for d in (start, end) if d) and (not start or not end or start <= end)
    except ValueError:
        return False


expected = {}
for id_, (outcome, start, end, qualifier, note) in judged.items():
    if valid(outcome, start, end):
        for path in index[id_]["paths"]:
            expected[(id_, path)] = bounds(start, end) if outcome == "range" else (None, None)

got = {name: {(id_, path): results[name][(index[id_]["source"], path, index[id_]["text"])] for id_, path in expected} for name in PARSERS}
weight = lambda key: occurrences[(index[key[0]]["source"], key[1], index[key[0]]["text"])]

print(f"{len(judged)} judged, {malformed} malformed lines skipped, {len(judged) - len({k[0] for k in expected})} dropped as invalid\n")
print(f"{'parser':8} {'correct':>8} {'wrong':>6}   {'accuracy':>9} {'precision':>10} {'recall':>7}   {'weighted acc':>12} {'prec':>6} {'recall':>7}")
for name in PARSERS:
    line = f"{name:8}"
    for w in (lambda key: 1, weight):
        total = sum(w(k) for k in expected)
        correct = sum(w(k) for k in expected if got[name][k] == expected[k])
        produced = sum(w(k) for k in expected if got[name][k] != (None, None))
        judge_ranges = sum(w(k) for k in expected if expected[k] != (None, None))
        correct_ranges = sum(w(k) for k in expected if got[name][k] == expected[k] != (None, None))
        cells = (correct / total, correct_ranges / produced, correct_ranges / judge_ranges)
        line += f" {correct:>8} {total - correct:>6}   {cells[0]:9.1%} {cells[1]:10.1%} {cells[2]:7.1%}" if w(next(iter(expected))) == 1 and w is not weight else f"   {cells[0]:12.1%} {cells[1]:6.1%} {cells[2]:7.1%}"
    print(line)

9998 judged, 5 malformed lines skipped, 5 dropped as invalid

parser    correct  wrong    accuracy  precision  recall   weighted acc   prec  recall
python       4726   5312       47.1%      48.2%   47.6%          89.6%  91.9%   89.7%
scala        8437   1601       84.1%      97.1%   84.4%          94.7%  98.4%   94.7%
new          9721    317       96.8%      97.2%   97.4%          98.7%  98.8%   98.7%


In [7]:
PARSER = "new"
LIMIT = 40

for key in sorted((k for k in expected if got[PARSER][k] != expected[k]), key=lambda k: -weight(k))[:LIMIT]:
    id_, path = key
    outcome, start, end, qualifier, note = judged[id_]
    print(f"{index[id_]['text']!r}  [{path}, {weight(key)}x]")
    print(f"    judge   {show(expected[key]) if outcome == 'range' else outcome}{'  ' + repr(qualifier) if qualifier and qualifier != 'exact' else ''}{'  ' + note if note else ''}")
    print(f"    {PARSER:7} {show(got[PARSER][key])}")

'[1920?]'  [production, 260x]
    judge   1910-01-01 .. 1929-12-31  'approximate'
    new     1920-01-01 .. 1920-12-31
'[1990?]'  [production, 222x]
    judge   1980-01-01 .. 1999-12-31  'approximate'
    new     1990-01-01 .. 1990-12-31
'[1905?]'  [production, 214x]
    judge   1895-01-01 .. 1914-12-31  'approximate'
    new     1905-01-01 .. 1905-12-31
'[1923?]'  [production, 152x]
    judge   1913-01-01 .. 1932-12-31  'approximate'
    new     1923-01-01 .. 1923-12-31
'[1946?]'  [production, 133x]
    judge   1936-01-01 .. 1955-12-31  'approximate'
    new     1946-01-01 .. 1946-12-31
'[1967?]'  [production, 104x]
    judge   1957-01-01 .. 1976-12-31  'approximate'
    new     1967-01-01 .. 1967-12-31
'[1942?]'  [production, 92x]
    judge   1932-01-01 .. 1951-12-31  'approximate'
    new     1942-01-01 .. 1942-12-31
'[1919?]'  [production, 90x]
    judge   1909-01-01 .. 1929-12-31  'approximate'
    new     1919-01-01 .. 1919-12-31
'1798?]'  [production, 78x]
    judge   1788-01-01

In [8]:
# Get 100 random dates parsed by the judge to manually check
import random

random_ids = random.sample(sorted(judged.keys()), 100)

for i in random_ids:
    raw = index[i]
    ans = judged[i]
    print(f"{raw['source']:<7}", f"{raw['text']:<70}", f"{ans[1]} to {ans[2]} ({ans[3]})")

axiell  c 1920 - c 1972                                                        1910-01-01 to 1982-12-31 (approximate)
axiell  September 1971-January 1972                                            1971-09-01 to 1972-01-31 (exact)
marc    1844-1866.                                                             1844-01-01 to 1866-12-31 (exact)
marc    ca. 1755]                                                              1745-01-01 to 1764-12-31 (approximate)
marc    1832-1833.                                                             1832-01-01 to 1833-12-31 (exact)
axiell  Oct 1953                                                               1953-10-01 to 1953-10-31 (exact)
axiell  Aug-Oct 1990                                                           1990-08-01 to 1990-10-31 (exact)
axiell  February 2003-December 2012                                            2003-02-01 to 2012-12-31 (exact)
axiell  1959-1975                                                              1959-01-01 to

In [9]:
# Compare Scala vs new Python where they disagree

disagreements = [(k, v) for k, v in verdicts.items() if v != 'all agree' and show(results['scala'][k])[0] == '1']
for k, v in random.sample(disagreements, 1):
    print(f"{k[2]:<50}", f"{show(results['scala'][k]):<40}", show(results['new'][k]))


1800s                                              1800-01-01 .. 1899-12-31                 1800-01-01 .. 1809-12-31


In [10]:
# 2604 [1944]                                        1944-04-26 .. 1944-04-26                 1944-01-01 .. 1944-12-31
# MDCCVIII. [1708] [1717]                            1717-08-17 .. 1717-08-17                 1708-01-01 .. 1717-12-31
# 2002, c1999.                                       1989-01-01 .. 2008-12-31                 no range
# 1885-03                                            1885-01-01 .. 1883-12-31                 no range
# 5452.                                              5452-01-01 .. 5452-12-31                 no range


In [97]:
# Compare new vs judge where they disagree

disagreements = [k for k in expected if got["new"][k] != expected[k]]
print(len(disagreements))
for id_, path in random.sample(disagreements, 50):
    outcome = judged[id_][0]
    judge = show(expected[(id_, path)]) if outcome == "range" else outcome
    print(f"{index[id_]['text']:<50}", f"{judge:<40}", show(got["new"][(id_, path)]))


317
n.d. c. 1980s-1990s                                1970-01-01 .. 1999-12-31                 1980-01-01 .. 1999-12-31
Mense Augusto 1536.                                1536-08-01 .. 1536-08-31                 1536-01-01 .. 1536-12-31
1497 die 4 Aprilis.                                1497-04-04 .. 1497-04-04                 1497-01-01 .. 1497-12-31
M.DCC.LXXIV. [1774] [1786]                         ambiguous                                1774-01-01 .. 1786-12-31
March 4, --MDCCXCIII. [1793]                       1793-03-04 .. 1793-03-04                 1793-01-01 .. 1793-12-31
l653.                                              1653-01-01 .. 1653-12-31                 no range
An XIII--1804.                                     ambiguous                                1804-01-01 .. 1804-12-31
[1539?]                                            1529-01-01 .. 1548-12-31                 1539-01-01 .. 1539-12-31
[1793?]-1800.                                      1783-01-01 .. 1800-12-31 

In [99]:
# Compare scala vs judge where they disagree

disagreements = [k for k in expected if got["scala"][k] != expected[k]]
print(len(disagreements))
for id_, path in random.sample(disagreements, 50):
    outcome = judged[id_][0]
    judge = show(expected[(id_, path)]) if outcome == "range" else outcome
    print(f"{index[id_]['text']:<50}", f"{judge:<40}", show(got["scala"][(id_, path)]))


1629
[between 1897 and 1898?]                           1897-01-01 .. 1907-12-31                 1897-01-01 .. 1898-12-31
1955, 1963                                         ambiguous                                1955-01-01 .. 1963-12-31
1691 [1692]                                        1692-01-01 .. 1692-12-31                 no range
Printed in the Year MDCCXXXIV. [1734]              1734-01-01 .. 1734-12-31                 no range
Maria Theresa, 1740-1780.                          1740-01-01 .. 1780-12-31                 no range
l'an second de la liberté française [1790]         1790-01-01 .. 1790-12-31                 no range
Frederick William II, 1786-1797.                   1786-01-01 .. 1797-12-31                 no range
Viśū, Āvaṇi, [1881]                                1881-01-01 .. 1881-12-31                 no range
1959, 1962                                         ambiguous                                1959-01-01 .. 1962-12-31
1725/6 [i.e. 1726]                    